## 1. Auto-Initialization (Experimental)

> **Experimental Feature.** Uses LLM. It can make mistakes. **Always check `evaluator.py` and `arif_loop.py` manually.**

### What it does:
* **Generates Code:** Automatically builds `evaluator.py` and `arif_loop.py`.
* **Audits Bugs:** A second agent reviews the code to prevent flaws or cheating.
* **Sanity Check:** Runs a quick test to ensure the loop and code is runnable.
* **Log:** Standard output and error information can be found in arif_init_output.log

### Argument Breakdown:
* `--task_background`: Describes the dataset and the core task goal.
* `--your_idea_about_loop`: Directs the loop's behavior or research strategy.
* `--METRIC_prompt`: Defines the target metric for the agent to optimize.
* `--HOW_TO_RUN_YOUR_CODE`: The command to execute your workflow code and the evaluator which agent is going to build.
  * **Environment Lock:** Use absolute paths (e.g., `/home/.../bin/python`) to force the agent to use your specific Conda/virtual environment. Avoids path mismatch.
* `--DIAGNOSTIC_TIMEOUT`: Max seconds allowed for the baseline sanity check.
* `--max_retry`: Max attempts for the agent to self-fix initialization bugs.
* `--cli_type`: The LLM engine driving the setup process.

### Why do we need an Evaluator?
* **The Concept:** Evaluator = Judge. Your training code = Player.
* **Locked Standard:** The Judge is strictly **protected**. Future Research Agents cannot modify it. Standard stays fair.
* **Goal Alignment (CRITICAL):** Does the Judge score exactly what you actually want? Ensure the evaluation logic completely aligns with your real objective.
* **Ensure Compatibility:** Check how the Judge loads your model. If it hardcodes feature counts or model structures, the Agent cannot experiment with them. All modified models will crash on load.
* **Rich Feedback:** Make the Judge print diagnostic logs or save plots. Good feedback helps the Research Agent brainstorm better fixes.

### What is `README_for_agent.md`?
* **The Blueprint:** Short guide explaining how to use the `arif` library.
* **The Purpose:** Fed directly into the Initialization Agent as its system prompt.

### Task Background: Seoul Bike Demand
* **Dataset Source:** UCI Machine Learning Repository (Dataset ID: 560).
* **Predict (Target):** `Rented Bike Count` (Number of bikes rented per hour).
* **Inputs (Features):** * *Time & Date:* Hour, Date.
  * *Weather:* Temperature, Humidity, Wind speed, Visibility, Solar radiation, Rainfall, Snowfall.
  * *Context:* Seasons, Holiday, Functioning Day.

When you see `[READY]`, move to the next step.

In [14]:
%%bash
export PYTHONUNBUFFERED=1
python arif_init.py \
  --task_background "This is a regression task on the Seoul Bike Sharing Demand dataset." \
  --your_idea_about_loop "Basically follows Standard Code Pattern in @README_for_agent.md." \
  --METRIC_prompt "evaluator metric use MSE on the test set." \
  --HOW_TO_RUN_YOUR_CODE "/home/liumx/.conda/envs/llamacpp/bin/python train.py && /home/liumx/.conda/envs/llamacpp/bin/python evaluator.py" \
  --DIAGNOSTIC_TIMEOUT 900 \
  --max_retry 5 \
  --cli_type "gemini"


[Step 1] Setup Agent is building evaluator.py...
[AIAgent] Executing gemini...
Agent response length: 2111

[Step 2] Setup Agent is generating arif_loop.py...
[AIAgent] Executing gemini...
Agent response length: 1923

[Step 3] Reviewer Agent is auditing the generated files...
[AIAgent] Executing gemini...
Reviewer response length: 3808

[Step 4] Setup Agent is refining work based on critique...
[AIAgent] Executing gemini...
Agent response length: 1829

Diagnostic: Running '/home/liumx/.conda/envs/llamacpp/bin/python train.py && /home/liumx/.conda/envs/llamacpp/bin/python evaluator.py'...
  [OK] Found metric: evaluator_metric:171198.739

[Step 7] Setup Agent is summarizing the initialization process...
[AIAgent] Executing gemini...
Summary response length: 3450

=== Agent Summary ===
I have successfully initialized the research automation environment for the Seoul Bike Sharing Demand regression task. Below is a detailed summary of the modifications and the design of the core components

## 2. The Agent generated Autonomous Research Loop

> 💡 The Python code cell below is **only for inspection**. The actual loop is executed directly as a script file at next next cell.

In [ ]:
from arif import AutoResearch, AIAgent
import os

def main():
    # --- Configuration ---
    # AGENT_TIMEOUT: timeout for LLM calls. CMD_TIMEOUT: timeout for code execution.
    AGENT_TIMEOUT, CMD_TIMEOUT = 300, 900 
    
    main_prompt = (
        "This is a regression task on the Seoul Bike Sharing Demand dataset. "
        "The goal is to minimize the Mean Squared Error (MSE) on the test set. "
        "You should modify 'train.py' to improve the model performance. "
        "Potential improvements include:\n"
        "- Feature Engineering: Look at the 'Date' column; it is currently being one-hot encoded, which is suboptimal. Extract day, month, or season.\n"
        "- Model Selection: Explore different regressors beyond the baseline RandomForest.\n"
        "- Hyperparameter Tuning: Optimize the chosen model's parameters.\n\n"
        "SYSTEM CONSTRAINTS:\n"
        "1. Your script must save the final trained model to 'model.joblib' in the current directory. Never run the code, let me run the code.\n"
        "2. The evaluator will load this file and compute the metric on a hidden test set (the last 30% of rows).\n"
        "3. [Anti-Cheating] The 'train.py' script has been restricted to only access the first 70% of the dataset. Do not try to bypass this split. "
        "The 'evaluator_metric:' string is reserved for the evaluator; do not print it in 'train.py'."
    )
    
    # 1. Initialize AutoResearch with file protection
    # We protect the evaluator and setup files to ensure the agent only modifies the strategy/training code.
    log_name = "arif_LLM_response.log"
    ar = AutoResearch(
        project_root="./", 
        protected_files=["evaluator.py", "arif_init.py", "README_for_agent.md", "arif_loop.py"], 
        log_path=log_name
    )
    
    # Initialize the AIAgent with the task instructions
    agent = AIAgent(
        engine="gemini", 
        system_prompt=main_prompt,
        default_guard=ar.guard,
        default_timeout=AGENT_TIMEOUT,
        log_path=log_name
    )
    
    # Start or continue a research branch
    B, L, S = ar.new_branch()
    best_metric = 171198.739 # We want to minimize MSE

    # The EXACT execution command as specified by the user
    EVAL_CMD = "/home/liumx/.conda/envs/llamacpp/bin/python train.py && /home/liumx/.conda/envs/llamacpp/bin/python evaluator.py"

    for _ in range(10): # Iterate through 20 experiment steps
        with ar.enter_exp(B, L, S):
            # 2. Provide context from previous trials at the same level
            history_text = ar.get_history(B=B, L=L, if_improved=False, limit=3, as_text=True)

            print(f"\n--- Starting Experiment {B}.{L}.{S} ---")
            print("Generating hypothesis based on history...")
            hypothesis_prompt = (
                f"Previous experiment history:\n{history_text}\n"
                "Based on the code and previous results, propose a hypothesis to improve the model and reduce the MSE metric."
            )
            hypothesis = agent.ask(hypothesis_prompt, new_session=True)

            # 3. Modify-Run-Evaluate Cycle
            # ar.modify_and_run_loop handles the internal retries if the code fails to run.
            success, current_metric, _, _ = ar.modify_and_run_loop(
                agent, 
                modify_prompt=". Now modify 'train.py' based on the proposed hypothesis. Ensure it saves the model to 'model.joblib'.",
                eval_cmd=EVAL_CMD,
                metric_extract="evaluator_metric: ",
                best_metric=best_metric,
                max_trials=3,
                timeout=CMD_TIMEOUT,
                smaller_is_better=True # We are minimizing MSE
            )

            print(f"Experiment finished. Success: {success}, Metric (MSE): {current_metric}")
            summary = agent.ask(". Now summarize the modifications made and the resulting performance changes.")

            # 4. Save results to history and evolve the branch
            ar.save_history(metric=current_metric, if_improved=success, hypothesis=hypothesis, summary=summary)
            
            if success:
                # If the metric improved, move to the next Level
                best_metric, L, S = current_metric, L + 1, 1
            else:
                # If it didn't improve, increment the Step for the current Level
                S += 1

if __name__ == "__main__":
    main()


## 3. Post-Run Analysis & Breakthroughs

The Agent ran **10 experiments** and successfully broke through **2 major levels**, slashing MSE from 171k to 97k. 

* **The Log:** Level 1.1.1 made the first giant leap. Level 1.2.5 optimized it further. The 1.3.* experiments built on top of 1.2.5 but failed to break through yet.

### Level 1 Breakthrough: Exp 1.1.1 (MSE 171k → 117k)
* **Smart Dates:** Extracted Month, Day, Day of Week. Stopped treating dates as dumb text strings.
* **Cyclic Hours:** Encoded Hour into Sine/Cosine waves. Tells the model 23:00 and 00:00 are next to each other.
* **Model Upgrade:** Swapped basic RandomForest for HistGradientBoostingRegressor. Faster, stronger trees.

### Level 2 Breakthrough: Exp 1.2.5 (MSE 117k → 97k)
* **Domain Context:** Added `IsPeak` (rush hour) and `IsWeekend` flags.
* **Weather Cross:** Multiplied Temp × Humidity and Temp × Solar to capture compound weather effects.
* **Target Transform:** Applied `log1p` to bike counts. Stabilized variance and killed impossible negative bike predictions.
* **Smarter Encoding:** Changed categories to OrdinalEncoder. GBDT handles ordinal arrays much better than One-Hot.

### Stuck at a Level?
If the loop hits a wall and stops improving, use these two strategies:
* **Inject Human Insights:** Stop the loop. Update your main prompt with new engineering angles or constraints.
* **Activate Web Search:** Use prompt to let the Agent browse papers or GitHub to harvest breakthrough ideas.

In [6]:
%%bash
export PYTHONUNBUFFERED=1
python arif_loop.py


>>> Entering Experiment: exp1.1.1

--- Starting Experiment 1.1.1 ---
Generating hypothesis based on history...
[AIAgent] Executing gemini...
  Trial 1/3: Modifying code...
[AIAgent] Executing gemini...
  Trial 1/3: Running evaluation...
  Current evaluator_metric: : inf (Best: 171198.7390)
  No improvement.
  Trial 2/3: Modifying code...
[AIAgent] Executing gemini...
  Trial 2/3: Running evaluation...
  Current evaluator_metric: : 117740.8454 (Best: 171198.7390)
  SUCCESS: Improved from 171198.7390 to 117740.8454!
Experiment finished. Success: True, Metric (MSE): 117740.8454
[AIAgent] Executing gemini...

>>> Entering Experiment: exp1.2.1

--- Starting Experiment 1.2.1 ---
Generating hypothesis based on history...
[AIAgent] Executing gemini...
  Trial 1/3: Modifying code...
[AIAgent] Executing gemini...
  Trial 1/3: Running evaluation...
[AutoResearch WARNING] TIMEOUT_ERROR: Command timed out after 900s
  Current evaluator_metric: : inf (Best: 117740.8454)
  No improvement.
  Trial 2/